- DIA on ViT model 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

- Core ViT model for gray images (channel = 1)

In [ ]:
# ============================================================
# HFFH_ViT: Hybrid Freehand Imaging ViT (MobileViT-style)
# Approximation faithful to the printed architecture you gave
# Input:  (B, 1, H, W)
# Output: (B, 1, H, W)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------- Core transformer components ----------

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x):
        # x: (B, N, dim)
        return self.fn(self.norm(x))


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=8, dropout=0.0):
        """
        dim: token embedding dim (8 or 16 in HFFH_ViT)
        heads: 4, so 3 * heads * dim_head = 96 = to_qkv out_features
        dim_head: 8
        """
        super().__init__()
        inner_dim = dim_head * heads * 3  # for q, k, v
        self.heads = heads
        self.dim_head = dim_head

        self.to_qkv = nn.Linear(dim, inner_dim, bias=False)
        self.attend = nn.Softmax(dim=-1)
        self.to_out = nn.Sequential(
            nn.Linear(dim_head * heads, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        """
        x: (B, N, dim)
        """
        b, n, _ = x.shape
        qkv = self.to_qkv(x)  # (B, N, 3 * heads * dim_head)
        qkv = qkv.view(b, n, 3, self.heads, self.dim_head)
        q, k, v = qkv.unbind(dim=2)  # each: (B, N, heads, dim_head)

        # (B, heads, N, dim_head)
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 1, 3)
        v = v.permute(0, 2, 1, 3)

        scale = self.dim_head ** -0.5
        dots = torch.matmul(q, k.transpose(-1, -2)) * scale  # (B, heads, N, N)
        attn = self.attend(dots)
        out = torch.matmul(attn, v)                           # (B, heads, N, dim_head)

        # (B, N, heads * dim_head)
        out = out.permute(0, 2, 1, 3).contiguous().view(b, n, self.heads * self.dim_head)
        out = self.to_out(out)  # (B, N, dim)
        return out


class Transformer(nn.Module):
    def __init__(self, dim, depth, mlp_dim, heads=4, dim_head=8, dropout=0.0):
        """
        dim: token dimension (8 or 16)
        depth: number of transformer layers
        mlp_dim: hidden dim in FFN (paper uses ~2x or 4x)
        """
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                PreNorm(dim, Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)),
                PreNorm(dim, FeedForward(dim, mlp_dim, dropout=dropout)),
            ]))

    def forward(self, x):
        for attn, ff in self.layers:
            x = x + attn(x)
            x = x + ff(x)
        return x


# ---------- MobileViT-style blocks ----------

class MV2Block(nn.Module):
    """
    MobileNetV2-style depthwise separable block.
    Matches the printed pattern:
      Conv2d(C,C,3x3,groups=C) -> BN -> SiLU -> Conv2d(C,C or C->C') -> BN
    With residual if in==out and stride==1.
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=stride,
                      padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=False),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.use_res_connect = (stride == 1 and in_channels == out_channels)

    def forward(self, x):
        out = self.conv(x)
        if self.use_res_connect:
            out = out + x
        return out

class MobileViTBlock(nn.Module):
    """
    MobileViT-style block:
      conv1: 3x3 (C->C)
      conv2: 1x1 (C->transformer_dim)
      transformer over patches in token dim = transformer_dim
      conv3: 1x1 (transformer_dim->C)
      conv4: 3x3 on concat(local, global) (2C->C)

    patch_size controls how big each transformer patch is (e.g., (2,2) or (4,4)).
    """
    def __init__(self, in_channels, transformer_dim, depth,
                 patch_size=(2, 2), mlp_dim=None, heads=4, dim_head=8, dropout=0.0):
        super().__init__()
        ph, pw = patch_size
        self.patch_h = ph
        self.patch_w = pw
        self.transformer_dim = transformer_dim

        # Local conv
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1,
                      padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=False),
        )

        # Channel projection: C -> transformer_dim (8 or 16)
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels, transformer_dim, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(transformer_dim),
            nn.SiLU(inplace=False),
        )

        if mlp_dim is None:
            mlp_dim = transformer_dim * 2  # you can change to *4 if you want to mimic 4x layers

        self.transformer = Transformer(
            dim=transformer_dim,
            depth=depth,
            mlp_dim=mlp_dim,
            heads=heads,
            dim_head=dim_head,
            dropout=dropout,
        )

        # Back to conv space: transformer_dim -> in_channels
        self.conv3 = nn.Sequential(
            nn.Conv2d(transformer_dim, in_channels, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=False),
        )

        # Fuse local+global: (2C -> C)
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=3, stride=1,
                      padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=False),
        )

    def forward(self, x):
        # x: (B, C, H, W)
        local_x = self.conv1(x)
        y = self.conv2(local_x)   # (B, transformer_dim, H, W)
        B, C, H, W = y.shape
        assert C == self.transformer_dim

        ph, pw = self.patch_h, self.patch_w

        # Pad so H,W divisible by patch size
        pad_h = (ph - H % ph) % ph
        pad_w = (pw - W % pw) % pw
        if pad_h != 0 or pad_w != 0:
            y = F.pad(y, (0, pad_w, 0, pad_h), mode="reflect")
            B, C, H, W = y.shape

        self._orig_hw = (H, W)

        # Unfold into non-overlapping patches
        # y_unfold: (B, C*ph*pw, L)
        y_unfold = F.unfold(y, kernel_size=(ph, pw), stride=(ph, pw))
        # reshape to (B, C, ph*pw, L) then average over patch pixels -> (B, C, L)
        y_unfold = y_unfold.view(B, C, ph * pw, -1).mean(dim=2)
        # tokens: (B, L, C)
        tokens = y_unfold.permute(0, 2, 1).contiguous()

        # Transformer over tokens
        tokens = self.transformer(tokens)  # (B, L, C)

        # Map tokens back to patches, broadcast to ph*pw pixels, fold back
        t = tokens.permute(0, 2, 1)                 # (B, C, L)
        t = t.unsqueeze(2).repeat(1, 1, ph * pw, 1) # (B, C, ph*pw, L)
        t = t.view(B, C * ph * pw, -1)              # (B, C*ph*pw, L)
        y = F.fold(t, output_size=(H, W), kernel_size=(ph, pw), stride=(ph, pw))

        # Crop to original (if we padded)
        orig_H, orig_W = self._orig_hw
        y = y[:, :, :orig_H, :orig_W]

        # Project back and fuse with local conv features
        y = self.conv3(y)
        out = torch.cat((x, y), dim=1)
        out = self.conv4(out)
        return out

# ---------- HFFH_ViT full network ----------

class HFFH_ViT(nn.Module):
    """
    Hybrid Freehand Imaging ViT
    - Input:  (B, 1, H, W)
    - Output: (B, 1, H, W)
    Matches the high-level structure you pasted:
      conv1: 1->16
      stem:  4x MV2Block(16->16)
      trunk:
        Stage1: MV2(16->16) + MobileViTBlock(16,8,depth=2)
        Stage2: MV2(16->16) + MobileViTBlock(16,8,depth=4)
        Stage3: MV2(16->32) + MobileViTBlock(32,16,depth=3)
      head:  1x1 conv 32->1
    """
    def __init__(self, img_channels=1, patch_size=(2, 2), dropout=0.0):
        super().__init__()

        # Initial conv: 1 -> 16
        self.conv1 = nn.Sequential(
            nn.Conv2d(img_channels, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.SiLU(inplace=False),
        )

        # Stem: 4× MV2Block(16->16)
        self.stem = nn.ModuleList([
            MV2Block(16, 16, stride=1),
            MV2Block(16, 16, stride=1),
            MV2Block(16, 16, stride=1),
            MV2Block(16, 16, stride=1),
        ])

        # Trunk: 3 stages
        self.trunk = nn.ModuleList()

        # Stage 1: channels 16, transformer_dim=8, depth=2 (as in printout)
        stage1 = nn.ModuleList([
            MV2Block(16, 16, stride=1),
            MobileViTBlock(
                in_channels=16,
                transformer_dim=8,
                depth=2,
                patch_size=patch_size,
                mlp_dim=16,      # 2x expansion for dim=8
                heads=4,
                dim_head=8,
                dropout=dropout,
            )
        ])
        self.trunk.append(stage1)

        # Stage 2: channels 16, transformer_dim=8, depth=4
        stage2 = nn.ModuleList([
            MV2Block(16, 16, stride=1),
            MobileViTBlock(
                in_channels=16,
                transformer_dim=8,
                depth=4,
                patch_size=patch_size,
                mlp_dim=32,      # 4x expansion for dim=8
                heads=4,
                dim_head=8,
                dropout=dropout,
            )
        ])
        self.trunk.append(stage2)

        # Stage 3: MV2 16->32, transformer_dim=16, depth=3
        stage3 = nn.ModuleList([
            MV2Block(16, 32, stride=1),
            MobileViTBlock(
                in_channels=32,
                transformer_dim=16,
                depth=3,
                patch_size=patch_size,
                mlp_dim=64,      # 4x expansion for dim=16
                heads=4,
                dim_head=8,
                dropout=dropout,
            )
        ])
        self.trunk.append(stage3)

        # ------------------ Lhamo: grayscale but SiLU negative values
        '''
        # Head: 32 -> 1
        self.condense_channels = nn.Sequential(
            nn.Sequential(
                nn.Conv2d(32, 1, kernel_size=1, stride=1, bias=False),
                nn.BatchNorm2d(1),
                nn.SiLU(inplace=False),
            )
        )
        '''
        # ------------------ Lhamo: ReLU For positive output pixel
        self.condense_channels = nn.Sequential(
            nn.Conv2d(32, 1, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(1),
            nn.ReLU(inplace=False),
            )

    def forward(self, x):
        # x: (B,1,H,W)
        x = self.conv1(x)  # (B,16,H,W)

        for block in self.stem:
            x = block(x)    # (B,16,H,W)

        # Trunk stages
        for stage in self.trunk:
            mv2, mvblock = stage
            x = mv2(x)
            x = mvblock(x)

        # Now 32 channels -> 1
        out = self.condense_channels(x)
        return out


- Matched Filter Algorithm (differentiable)

In [ ]:
#======================================= MFA definitions
@dataclass
class MFAParams:
    F0: float          # start freq (Hz), e.g. 77e9
    c0: float          # lightspeed, e.g. 3e8
    dx: float          # mm
    dy: float          # mm
    z0_mm: float       # target range (mm)
    bbox: tuple        # (xmin, xmax, ymin, ymax) in mm
    FS: float          # sampling rate (Hz)
    K0: float          # chirp slope (Hz/s)
    tI: float          # instrument delay (s)
    nFFTtime: int      # range FFT size, e.g. 1024
    nFFTspace: int     # spatial FFT size, e.g. 1024


def build_matched_filter(params: MFAParams, device, dtype):
    """
    PyTorch version of refMF(params) in MATLAB.
    Returns: matchedFilter (nFFTspace x nFFTspace), complex tensor.
    """
    nF = params.nFFTspace
    dx_m = params.dx * 1e-3
    dy_m = params.dy * 1e-3
    z0_m = params.z0_mm * 1e-3

    # Index grid like MATLAB: (-(nF-1)/2 : (nF-1)/2) * dx
    idx = torch.arange(nF, device=device, dtype=dtype) - (nF / 2 - 0.5)
    x = dx_m * idx                          # (nF,)
    y = dy_m * idx.view(-1, 1)              # (nF,1) -> broadcast

    k = 2.0 * torch.pi * params.F0 / params.c0   # wavenumber
    R = torch.sqrt(x**2 + y**2 + z0_m**2)        # (nF, nF)

    phase = -1j * 2.0 * k * R                    # -j 2 k R
    matched_filter = torch.exp(phase)            # complex
    return matched_filter


def mfa_image_from_slice(sarData: torch.Tensor, params: MFAParams):
    """
    Differentiable PyTorch version of dlMFA.
    sarData: (M, N) complex tensor (after range FFT + serpentine)
    Returns:
        img_mag_norm: (B, A) real tensor, RMS-normalized magnitude
        img_complex : (B, A) complex tensor
    """
    assert torch.is_complex(sarData), "sarData must be complex (torch.complex64/128)"
    device = sarData.device
    dtype  = sarData.real.dtype

    # Build matched filter on same device/dtype
    mf = build_matched_filter(params, device=device, dtype=dtype)  # (nF, nF)

    yPointM, xPointM = sarData.shape
    yPointF, xPointF = mf.shape   # both nFFTspace

    # --- zero-pad sarData up to matched filter size (like MATLAB) ---
    pad_y_pre = max((yPointF - yPointM) // 2, 0)
    pad_y_post = max(yPointF - yPointM - pad_y_pre, 0)
    pad_x_pre = max((xPointF - xPointM) // 2, 0)
    pad_x_post = max(xPointF - xPointM - pad_x_pre, 0)

    if pad_y_pre > 0 or pad_y_post > 0:
        pad_y_top = torch.zeros(pad_y_pre, xPointM, dtype=sarData.dtype, device=device)
        pad_y_bot = torch.zeros(pad_y_post, xPointM, dtype=sarData.dtype, device=device)
        sarData = torch.cat([pad_y_top, sarData, pad_y_bot], dim=0)

    if pad_x_pre > 0 or pad_x_post > 0:
        y_curr = sarData.shape[0]
        pad_x_left  = torch.zeros(y_curr, pad_x_pre, dtype=sarData.dtype, device=device)
        pad_x_right = torch.zeros(y_curr, pad_x_post, dtype=sarData.dtype, device=device)
        sarData = torch.cat([pad_x_left, sarData, pad_x_right], dim=1)

    # 2D FFT of data and matched filter
    sar_fft = torch.fft.fft2(sarData)
    mf_fft  = torch.fft.fft2(mf)

    # Convolution in freq domain, then ifft2
    img_shifted = torch.fft.ifft2(sar_fft * mf_fft)

    # fftshift (2D)
    def fftshift2d(x):
        h, w = x.shape[-2:]
        return torch.roll(torch.roll(x, shifts=h//2, dims=-2),
                          shifts=w//2, dims=-1)

    img = fftshift2d(img_shifted)

    # Crop using bbox like MATLAB
    J, I = img.shape
    bbox = params.bbox
    dx = params.dx
    dy = params.dy

    xij0 = round(bbox[0] / dx - 0.5 + I / 2.0)
    xij1 = round(bbox[1] / dx - 0.5 + I / 2.0)
    ykl0 = round(bbox[2] / dy - 0.5 + J / 2.0)
    ykl1 = round(bbox[3] / dy - 0.5 + J / 2.0)

    x0 = max(int(xij0), 0)
    x1 = min(int(xij1), I-1)
    y0 = max(int(ykl0), 0)
    y1 = min(int(ykl1), J-1)

    img_cropped = img[y0:y1+1, x0:x1+1]         # (B, A) complex

    # fliplr
    img_complex = torch.flip(img_cropped, dims=[1])

    img_mag = torch.abs(img_complex)
  
    return img_mag, img_complex

- Victim Imaging/Forward imaging: (1) MFA and (2) MFA+VIT

    imaging chain (CH1): rawSAR -> MFA -> resize -> ViT -> || [DIA] 
- Generate Target Image

In [ ]:
import torch
import torch.nn.functional as F

# ---------- ------------------------- Victim forward imaging model 
def victim_forward(raw_cube_adv: torch.Tensor, params: MFAParams, model: torch.nn.Module, mfa_or_vit: str):
    """
    raw_cube_adv: (Nsamp, M, N) complex tensor
    returns:
        if mfa_or_vit == 'mfa': (1,1,H,W) normalized MFA
        if mfa_or_vit == 'vit': (1,1,256,256) Mobile-ViT output
    """
    # Range FFT along fast time (dim=0)
    raw_fft = torch.fft.fft(raw_cube_adv, n=params.nFFTtime, dim=0)  # (nFFTtime,M,N)

    # Compute k0_range_bin
    z0_m = params.z0_mm * 1e-3
    k0_range_bin = round(
        params.K0 / params.FS * (2.0 * z0_m / params.c0 + params.tI) * params.nFFTtime
    )
    k0_range_bin = int(k0_range_bin)

    # Extract slice
    sar_slice = raw_fft[k0_range_bin, :, :]   # (M,N) complex

    # Serpentine correction
    sar_slice = sar_slice.clone()
    sar_slice[1::2, :] = torch.flip(sar_slice[1::2, :], dims=[1])

    # MFA imaging -> magnitude
    mfa_mag, _ = mfa_image_from_slice(sar_slice, params)  # (H,W)
    #----------------------------------------- mfa or ViT
    if mfa_or_vit == 'mfa':
        x = mfa_mag.unsqueeze(0).unsqueeze(0) # (1,1,H,W)
        sr_out = x  # (1,1,H,W)

    elif mfa_or_vit == 'vit':
        #x =  (mfa_mag - mfa_mag.min()) / (mfa_mag.max() - mfa_mag.min() + 1e-12)
        #x = x.unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

        mx = mfa_mag.amax().detach() + 1e-12
        x  = (mfa_mag / mx).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

        # Resize to 256×256 for Mobile-ViT
        x256 = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)  # (1,1,256,256)
        sr_out = model(x256)  # (1,1,256,256)

    else:
        raise ValueError("mfa_or_vit must be either 'mfa' or 'vit'")
    return sr_out


## Target and Clean Image Generation

In [ ]:
import os
import torch
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from scipy.io import loadmat

# -----------------------------
# Paths / run configuration
# -----------------------------
data_dir         = os.path.join(os.getcwd(), "data")          # model weights, D.mat, etc.
raw_sar_data_dir = os.path.join(os.getcwd(), "raw_sar_data")  # holds knife.mat, plier.mat, ...

mfa_or_vit  = "vit"     # "mfa" or "vit"
target_mode = "object"  # "noise" or "object"


target_raw_select = 1   # target: 1..10 (used only if target_mode == "object")

rawData_select    = 6   # victim: 1..10


# ============================================================
# Attack hyperparameters
# ============================================================
num_iters  = 1000
lambda_L2  = 1e-3


lr         = 1e-1 ###########################################################################


# ============================================================
# Constraint toggles
# ============================================================
use_amax_projection  = False
Amax                 = 2.0

use_pa_pr_projection = True  
PaPr_max_dB          = -10.0
PaPr_max             = 10 ** (PaPr_max_dB / 10.0)

In [ ]:
# Dataset names (must match .mat filenames in raw_sar_data_dir)
rawData = [
    "knife",                 # 1
    "plier",                 # 2
    "scissor",               # 3
    "screw_driver",          # 4
    "sharp_paint_speader",   # 5
    "dragger",               # 6
    "wrench",                # 7
    "gun",                   # 8
    "rifle",                 # 9
    "butcher_knife",               # 10
]


VICTIM_GEOM = {
    "knife":               (1.0, 1.0, 185.0, 5_000_000.0),
    "plier":               (1.0, 2.0, 210.0, 5_000_000.0),
    "scissor":             (1.0, 2.0, 215.0, 5_000_000.0),
    "screw_driver":        (1.0, 2.0, 230.0, 5_000_000.0),
    "sharp_paint_speader": (1.0, 2.0, 180.0, 5_000_000.0),
    "dragger":             (1.0, 2.0, 195.0, 5_000_000.0),

    # MATLAB special case:
    "wrench":              (1.0, 1.0, 170.0, 9_121_000.0),

    "gun":                 (1.0, 1.0, 185.0, 9_121_000.0),
    "rifle":               (1.0, 1.0, 185.0, 9_121_000.0),
    "butcher_knife":             (1.0, 2.0, 210.0, 5_000_000.0),
}

TARGET_GEOM = {
    "knife":               (1.0, 1.0, 185.0, 5_000_000.0),
    "plier":               (1.0, 2.0, 220.0, 5_000_000.0),

    # MATLAB target says scissor z0_tgt = 215
    "scissor":             (1.0, 2.0, 215.0, 5_000_000.0),

    "screw_driver":        (1.0, 2.0, 230.0, 5_000_000.0),
    "sharp_paint_speader": (1.0, 2.0, 180.0, 5_000_000.0),
    "dragger":             (1.0, 2.0, 195.0, 5_000_000.0),

    # MATLAB special case:
    "wrench":              (1.0, 1.0, 180.0, 9_121_000.0),

    # MATLAB target for gun: (dx_t=1, dy_t=2, z0_tgt=185)
    "gun":                 (1.0, 1.0, 185.0, 9_121_000.0),

    "rifle":               (1.0, 1.0, 185.0, 9_121_000.0),
    "butcher_knife":             (1.0, 2.0, 210.0, 5_000_000.0),
}

# -----------------------------
# Load trained ViT model
# -----------------------------
model = HFFH_ViT(img_channels=1, patch_size=(4, 4), dropout=0.0).to(device)
model.load_state_dict(
    torch.load(os.path.join(data_dir, "hffh_vit_ch1_best_epoch_015.pth"), map_location=device)
)
model.eval()

# -----------------------------
# Load victim data
# -----------------------------
victim_name = rawData[rawData_select - 1]
victim_path = os.path.join(raw_sar_data_dir, f"{victim_name}.mat")

raw_cube = torch.as_tensor(
    sio.loadmat(victim_path)["adcDataCube"],
    device=device
).to(torch.complex64)

Nsamp, M, N = raw_cube.shape
Np = M*N

# --- MATLAB-matched victim geometry (including FS) ---
if victim_name not in VICTIM_GEOM:
    raise ValueError(f"Unknown victim selection: {victim_name}")
dx, dy, z0_mm, FS_v = VICTIM_GEOM[victim_name]

print(
    f"victim={victim_name} | raw_cube shape: {tuple(raw_cube.shape)} | "
    f"dx={dx} dy={dy} z0_mm={z0_mm} FS={FS_v/1e6:.3f} MHz"
)

params = MFAParams(
    F0        = 77e9,
    c0        = 3e8,
    dx        = float(dx),
    dy        = float(dy),
    z0_mm     = float(z0_mm),
    bbox      = (-200.0, 200.0, -200.0, 200.0),
    FS        = float(FS_v),            # <-- UPDATED: MATLAB-style FS
    K0        = 70.295e12,
    tI        = 4.5225e-10,
    nFFTtime  = 1024,
    nFFTspace = 1024,
)

# -------------------------------
# CLEAN image (forward pass #1)
# -------------------------------
with torch.no_grad():
    clean_img = victim_forward(raw_cube, params, model, mfa_or_vit)  # (1,1,H,W)

# shared global scale from clean image
#global_scale = clean_img.abs().amax() + 1e-12
#clean_img = clean_img / global_scale

# -------------------------------
# TARGET image (forward pass #2)
# -------------------------------
if target_mode.lower() == "noise":
    # Keep your old noise behavior (minimal change):
    # shuffle across aperture for each fast-time sample
    Np = M * N
    g = torch.Generator(device=raw_cube.device).manual_seed(42)

    raw_flat = raw_cube.reshape(Nsamp, Np)
    raw_shuf = raw_flat.clone()
    for t in range(Nsamp):
        perm = torch.randperm(Np, generator=g, device=raw_cube.device)
        raw_shuf[t] = raw_shuf[t, perm]
    raw_shuf = raw_shuf.reshape(Nsamp, M, N)

    with torch.no_grad():
        target_img = victim_forward(raw_shuf, params, model, mfa_or_vit)
        # shared scaling
        #target_img = target_img / global_scale

elif target_mode.lower() == "object":
    target_name = rawData[target_raw_select - 1]
    target_path = os.path.join(raw_sar_data_dir, f"{target_name}.mat")

    raw_cube_tgt = torch.as_tensor(
        sio.loadmat(target_path)["adcDataCube"],
        device=device
    ).to(torch.complex64)

    print(f"target={target_name} | raw_cube_tgt shape: {tuple(raw_cube_tgt.shape)}")

    # --- MATLAB-matched target geometry (including FS) ---
    if target_name not in TARGET_GEOM:
        raise ValueError(f"Unknown target selection: {target_name}")
    dx_t, dy_t, z0_t_mm, FS_t = TARGET_GEOM[target_name]

    params_tgt = MFAParams(
        F0        = params.F0,
        c0        = params.c0,
        dx        = float(dx_t),
        dy        = float(dy_t),
        z0_mm     = float(z0_t_mm),
        bbox      = params.bbox,
        FS        = float(FS_t),         # <-- UPDATED: MATLAB-style target FS
        K0        = params.K0,
        tI        = params.tI,
        nFFTtime  = params.nFFTtime,
        nFFTspace = params.nFFTspace,
    )

    print(
        f"target params: dx={params_tgt.dx}, dy={params_tgt.dy}, "
        f"z0_mm={params_tgt.z0_mm}, FS={params_tgt.FS/1e6:.3f} MHz"
    )

    with torch.no_grad():
        target_img = victim_forward(raw_cube_tgt, params_tgt, model, mfa_or_vit)
        # shared scaling
        #target_img = target_img / global_scale

else:
    raise ValueError("target_mode must be 'noise' or 'object'.")

# If output sizes differ, resize target to match clean for plotting/comparisons
if target_img.shape != clean_img.shape:
    target_img = F.interpolate(
        target_img,
        size=clean_img.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

# ------------------------------------------------------------
# ViT version: Load D_1 / D_2 and build D_aa, then sample to D_t
# EXACTLY mirrors your MATLAB branching + rng behavior
# ------------------------------------------------------------

# ---- Load D_1 (required)
D1_np = loadmat(os.path.join(data_dir, "D_1.mat"))["D"]  # (Nsamp_pool, K1)
D_1_t = torch.as_tensor(D1_np.astype(np.complex64), dtype=torch.complex64, device=device)

# ---- Load D_2 (optional)
D_2_t = None
d2_path = os.path.join(data_dir, "D_2.mat")
if os.path.exists(d2_path):
    D2_np = loadmat(d2_path)["D"]  # (Nsamp_pool, K2)
    D_2_t = torch.as_tensor(D2_np.astype(np.complex64), dtype=torch.complex64, device=device)


if (Nsamp == 512) and ((M * N) > 40000) and (D_2_t is not None):
    D_temp = torch.cat([D_1_t, D_2_t], dim=0)      # stack rows
    D_aa   = torch.cat([D_temp, D_temp], dim=1)    # tile cols (duplicate columns)

elif (Nsamp == 512) and ((M * N) == 40000) and (D_2_t is not None):
    D_aa = torch.cat([D_1_t, D_2_t], dim=0)        # stack rows

else:
    D_aa = D_1_t

# ------------------------------------------------------------
# Sample columns to build D_pool, then pick one column per aperture
#   targetK = M*N
#   rng(0)
#   sample_idx = randi(size(D_aa,2), [1,targetK])
#   D_pool = D_aa(:, sample_idx)           -> (Nsamp_pool, targetK)
#   sel_idx  = randi(size(D_pool,2), [1,Np])
#   D_t      = D_pool(:, sel_idx)          -> (Nsamp_pool, Np)
# ------------------------------------------------------------
targetK = M * N

# MATLAB rng(0) equivalent for this usage pattern (deterministic)
rng = np.random.default_rng(0)

K_pool = D_aa.shape[1]
sample_idx = rng.integers(low=0, high=K_pool, size=targetK, endpoint=False)
D_pool = D_aa[:, sample_idx]                       # (Nsamp_pool, targetK)
sel_idx = rng.integers(low=0, high=D_pool.shape[1], size=Np, endpoint=False)
D_t = D_pool[:, sel_idx]                              # (Nsamp_pool, Np)

# ---- Make it match (Nsamp, M, N) for ViT A(m,n) attack ----
# Use first Nsamp rows if pool has more rows (common)
if D_t.shape[0] < Nsamp:
    raise RuntimeError(f"D_t has fewer rows than Nsamp: {D_t.shape[0]} < {Nsamp}")
D_t = D_t[:Nsamp, :]                                  # (Nsamp, Np)

D_cube = D_t.reshape(Nsamp, M, N)                     # (Nsamp, M, N)

print(f"[D] D_t shape   : {tuple(D_t.shape)} (Nsamp, Np)")
print(f"[D] D_cube shape: {tuple(D_cube.shape)} (Nsamp, M, N)")

#sel_idx = rng.integers(low=0, high=D_pool.shape[1], size=Np, endpoint=False)
#D_cube = D_pool[:, sel_idx]                           # (Nsamp_pool, Np)

print(f"[D] D_1 shape   : {tuple(D_1_t.shape)}")
print(f"[D] D_2 shape   : {None if D_2_t is None else tuple(D_2_t.shape)}")
print(f"[D] D_aa shape  : {tuple(D_aa.shape)}")
print(f"[D] D_pool shape: {tuple(D_pool.shape)} (Nsamp_pool, targetK)")
print(f"[D] D_t shape   : {tuple(D_cube.shape)} (Nsamp_pool, Np)")


print("clean_img  |.| min/max:", clean_img.abs().min().item(), "/", clean_img.abs().max().item())
print("target_img |.| min/max:", target_img.abs().min().item(), "/", target_img.abs().max().item())
print("clean_img :", tuple(clean_img.shape), clean_img.dtype, clean_img.device)
print("target_img:", tuple(target_img.shape), target_img.dtype, target_img.device)

# ---- quick visualization
clean2d  = clean_img.detach().abs().squeeze().cpu().numpy()
target2d = target_img.detach().abs().squeeze().cpu().numpy()

plt.figure(figsize=(10, 4))

ax1 = plt.subplot(1, 2, 1)
im1 = ax1.imshow(clean2d, cmap="gray", origin="lower")
ax1.set_title("Clean (vit)")
ax1.axis("off")
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

ax2 = plt.subplot(1, 2, 2)
im2 = ax2.imshow(target2d, cmap="gray", origin="lower")
ax2.set_title(f"Target ({target_mode})")
ax2.axis("off")
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## DIA Optimization

In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F

EPS = 1e-12

# Reference victim power (raw cube domain)
Pr_ref = (torch.linalg.norm(raw_cube) ** 2 + EPS).item()

# ============================================================
# Optimization variables
# ============================================================
A_re = torch.nn.Parameter(1e-3 * torch.randn(M, N, device=device))
A_im = torch.nn.Parameter(1e-3 * torch.randn(M, N, device=device))
optimizer = torch.optim.Adam([A_re, A_im], lr=lr)

# ============================================================
# Early-stop knobs: MATLAB-style diminishing returns
# ============================================================
maxIter_cap  = num_iters
check_every  = 25
W_win        = 100
rel_drop_min = 1e-3
abs_drop_min = 0.0
K_weak       = 3
weakWinCount = 0

# ============================================================
# History trackers
# ============================================================
loss_hist   = []
meanA_hist  = []
maxA_hist   = []
PaPr_hist   = []
PaPrdB_hist = []

bestLoss = float("inf")
bestIter = 0

A_re_best = A_re.detach().clone()
A_im_best = A_im.detach().clone()

print(
    f"\nOptimizing complex gain A over aperture grid (M={M}, N={N}, Np={M*N}) "
    f"with victim pipeline: {mfa_or_vit} | lr={lr:.3g} | lambda_L2={lambda_L2:.3g} | "
    f"Pa/Pr<={PaPr_max_dB:.2f} dB ({int(use_pa_pr_projection)}) | "
    f"|A|<={Amax:.3g} ({int(use_amax_projection)})\n"
)

iter = 0
while True:
    if iter >= maxIter_cap:
        print(f"Reached maxIter cap ({maxIter_cap}).")
        break

    iter += 1

    # =========================================================
    # 1) Forward / gradients through full pipeline
    # =========================================================
    optimizer.zero_grad()

    A_raw = torch.complex(A_re, A_im)              # (M,N)
    delta = D_cube * A_raw.unsqueeze(0)            # (Nsamp,M,N)
    raw_adv = raw_cube + delta                     # (Nsamp,M,N)

    adv_img = victim_forward(raw_adv, params, model, mfa_or_vit)
    #adv_img = adv_img / global_scale

    loss_img = F.mse_loss(adv_img, target_img)
    loss_reg = lambda_L2 * torch.mean(torch.abs(A_raw) ** 2)
    loss = loss_img + loss_reg

    loss.backward()
    optimizer.step()

    # =========================================================
    # 2) Projection(s) on UPDATED A
    # =========================================================
    with torch.no_grad():
        A_num = torch.complex(A_re, A_im)

        # -----------------------------------------------------
        # Projection 1: |A| <= Amax
        # -----------------------------------------------------
        if use_amax_projection:
            mags = torch.abs(A_num)
            scale_amax = torch.ones_like(mags)
            over = mags > Amax
            scale_amax[over] = Amax / (mags[over] + EPS)
            A_num = A_num * scale_amax

        # -----------------------------------------------------
        # Projection 2: enforce Pa/Pr <= PaPr_max
        # -----------------------------------------------------
        delta_now = D_cube * A_num.unsqueeze(0)
        Pa_now = (torch.linalg.norm(delta_now) ** 2).item()
        PaPr_now = Pa_now / Pr_ref

        if use_pa_pr_projection and (PaPr_now > PaPr_max):
            scale_pow = math.sqrt(PaPr_max / (PaPr_now + EPS))
            A_num = A_num * scale_pow

            delta_now = D_cube * A_num.unsqueeze(0)
            Pa_now = (torch.linalg.norm(delta_now) ** 2).item()
            PaPr_now = Pa_now / Pr_ref

        PaPr_now_dB = 10.0 * math.log10(PaPr_now + EPS)

        # write back projected A
        A_re.copy_(A_num.real)
        A_im.copy_(A_num.imag)

        meanA_now = torch.mean(torch.abs(A_num)).item()
        maxA_now  = torch.max(torch.abs(A_num)).item()

    # =========================================================
    # 3) Evaluate POST-projection loss (feasible iterate)
    # =========================================================
    with torch.no_grad():
        A_con = torch.complex(A_re, A_im)
        delta_con = D_cube * A_con.unsqueeze(0)
        raw_con = raw_cube + delta_con

        adv_img2 = victim_forward(raw_con, params, model, mfa_or_vit)
        #adv_img2 = adv_img2 / global_scale

        loss_img2 = F.mse_loss(adv_img2, target_img)
        loss_reg2 = lambda_L2 * torch.mean(torch.abs(A_con) ** 2)
        loss_post = loss_img2 + loss_reg2
        lossVal = float(loss_post.item())

        mse_AT = loss_img2.item()
        mse_AC = F.mse_loss(adv_img2, clean_img).item()

        g_re = 0.0 if A_re.grad is None else torch.max(torch.abs(A_re.grad)).item()
        g_im = 0.0 if A_im.grad is None else torch.max(torch.abs(A_im.grad)).item()
        Gnow = max(g_re, g_im)

    # =========================================================
    # 4) Track stats
    # =========================================================
    loss_hist.append(lossVal)
    meanA_hist.append(meanA_now)
    maxA_hist.append(maxA_now)
    PaPr_hist.append(PaPr_now)
    PaPrdB_hist.append(PaPr_now_dB)

    # =========================================================
    # 5) Update best
    # =========================================================
    if lossVal < bestLoss:
        bestLoss = lossVal
        bestIter = iter
        A_re_best = A_re.detach().clone()
        A_im_best = A_im.detach().clone()

    # =========================================================
    # 6) Logging
    # =========================================================
    if (iter % 25 == 0) or (iter == 1) or (iter == maxIter_cap):
        print(
            f"Iter {iter:04d} | Loss={lossVal:.4e} (best={bestLoss:.4e} @{bestIter}) | "
            f"G={Gnow:.6e} | MSE(A,T)={mse_AT:.4e}, MSE(C,A)={mse_AC:.4e} | "
            f"E|A|={meanA_now:.4e}, max|A|={maxA_now:.4e} | "
            f"Pa/Pr={PaPr_now:.4e} ({PaPr_now_dB:.2f} dB)"
        )

    # =========================================================
    # 7) Early stop: diminishing returns over a window
    # =========================================================
    if iter > W_win and (iter % check_every == 0):
        L_old = loss_hist[iter - W_win - 1]
        L_new = loss_hist[iter - 1]

        rel_drop = (L_old - L_new) / max(abs(L_old), 1e-12)
        abs_drop = (L_old - L_new)

        if abs_drop_min > 0:
            isWeak = (rel_drop < rel_drop_min) and (abs_drop < abs_drop_min)
        else:
            isWeak = (rel_drop < rel_drop_min)

        if isWeak:
            weakWinCount += 1
        else:
            weakWinCount = 0

        if weakWinCount >= K_weak:
            print(
                f"Early stop: diminishing returns. Over last {W_win} iters: "
                f"rel_drop={rel_drop:.3e}, abs_drop={abs_drop:.3e}. "
                f"Triggered {weakWinCount}/{K_weak}."
            )
            break

# -------------------------
# Trim history arrays
# -------------------------
loss_hist   = np.array(loss_hist)
meanA_hist  = np.array(meanA_hist)
maxA_hist   = np.array(maxA_hist)
PaPr_hist   = np.array(PaPr_hist)
PaPrdB_hist = np.array(PaPrdB_hist)

# -------------------------
# Restore best A
# -------------------------
with torch.no_grad():
    A_re.copy_(A_re_best)
    A_im.copy_(A_im_best)

print("-----------------------------")
print(f"Done. Best loss {bestLoss:.4e} at iter {bestIter} (ran {iter} iters).")
print("-----------------------------")

## Evaluations

In [ ]:
import numpy as np
from matplotlib.patches import Rectangle
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

EPS = 1e-12

def get_roi_box(img2d, thr_frac=0.25, pad=6):
    """
    img2d: (H,W) numpy array
    returns: (r1, r2, c1, c2), inclusive
    """
    A = np.abs(img2d).astype(np.float64)
    A = A / (A.max() + EPS)

    rows = np.where(np.max(A, axis=1) > thr_frac)[0]
    cols = np.where(np.max(A, axis=0) > thr_frac)[0]

    if rows.size == 0 or cols.size == 0:
        r, c = np.unravel_index(np.argmax(A), A.shape)
        rows = np.array([r], dtype=int)
        cols = np.array([c], dtype=int)

    r1 = max(int(rows[0])  - pad, 0)
    r2 = min(int(rows[-1]) + pad, A.shape[0] - 1)
    c1 = max(int(cols[0])  - pad, 0)
    c2 = min(int(cols[-1]) + pad, A.shape[1] - 1)

    return (r1, r2, c1, c2)


def crop_box(img2d, box):
    r1, r2, c1, c2 = box
    return img2d[r1:r2+1, c1:c2+1]


def compute_metrics_roi(x, y):
    x = x.astype(np.float64)
    y = y.astype(np.float64)

    mse = np.mean((x - y) ** 2)

    num = np.sum(x * y)
    den = np.sqrt(np.sum(x**2) * np.sum(y**2)) + EPS
    ncc = num / den

    dr = float(y.max() - y.min()) + EPS

    if x.shape == y.shape and min(x.shape) >= 11:
        ssim_val = ssim(x, y, data_range=dr)
    else:
        ssim_val = np.nan

    if x.shape == y.shape:
        psnr_val = psnr(y, x, data_range=dr)
    else:
        psnr_val = 10.0 * np.log10((dr**2) / (mse + EPS))

    return {
        "mse": mse,
        "ncc": ncc,
        "ssim": ssim_val,
        "psnr": psnr_val,
    }


def draw_roi_box(ax, box, color="r", lw=2):
    r1, r2, c1, c2 = box
    rect = Rectangle(
        (c1, r1),
        c2 - c1 + 1,
        r2 - r1 + 1,
        fill=False,
        edgecolor=color,
        linewidth=lw
    )
    ax.add_patch(rect)


# ------------------------------------------------------------
# ROI setup for object-mode evaluation
# ------------------------------------------------------------
roi_thr = 0.80
roi_pad = 6

clean2d_for_roi  = clean_img.detach().abs().squeeze().cpu().numpy()
target2d_for_roi = target_img.detach().abs().squeeze().cpu().numpy()

if target_mode.lower() == "object":
    roi_box_target = get_roi_box(target2d_for_roi, thr_frac=roi_thr, pad=roi_pad)
    roi_box_clean  = get_roi_box(clean2d_for_roi,  thr_frac=roi_thr, pad=roi_pad)

    print(f"Target ROI [r1 r2 c1 c2] = {roi_box_target}")
    print(f"Clean  ROI [r1 r2 c1 c2] = {roi_box_clean}")
else:
    roi_box_target = (0, target2d_for_roi.shape[0]-1, 0, target2d_for_roi.shape[1]-1)
    roi_box_clean  = (0, clean2d_for_roi.shape[0]-1,  0, clean2d_for_roi.shape[1]-1)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

EPS = 1e-12

with torch.no_grad():
    # ---- reconstruct attacked measurements (cube domain)
    A_opt = torch.complex(A_re, A_im)                       # (M,N)
    delta_opt = D_cube * A_opt.unsqueeze(0)                 # (Nsamp,M,N)
    raw_adv_opt = raw_cube + delta_opt                      # (Nsamp,M,N)

    # ---- reconstruct attacked image via victim pipeline
    adv_img = victim_forward(raw_adv_opt, params, model, mfa_or_vit)  # (1,1,H,W)

# -------------------------
# Convert to numpy magnitude images
# -------------------------
adv2d = adv_img.detach().abs().squeeze().cpu().numpy()
tgt2d = target_img.detach().abs().squeeze().cpu().numpy()
cln2d = clean_img.detach().abs().squeeze().cpu().numpy()

# -------------------------
# ROI crops
# -------------------------
adv_t = crop_box(adv2d, roi_box_target)
tgt_t = crop_box(tgt2d, roi_box_target)

adv_c = crop_box(adv2d, roi_box_clean)
cln_c = crop_box(cln2d, roi_box_clean)

# -------------------------
# ROI metrics
# -------------------------
AT = compute_metrics_roi(adv_t, tgt_t)
AC = compute_metrics_roi(adv_c, cln_c)

mse_AT  = AT["mse"]
ncc_AT  = AT["ncc"]
ssim_AT = AT["ssim"]
psnr_AT = AT["psnr"]

mse_AC  = AC["mse"]
ncc_AC  = AC["ncc"]
ssim_AC = AC["ssim"]
psnr_AC = AC["psnr"]

# -------------------------
# Global Pa/Pr (raw domain)
# -------------------------
with torch.no_grad():
    Pa = (torch.linalg.norm(delta_opt) ** 2).item()
    Pr = (torch.linalg.norm(raw_cube) ** 2).item() + EPS
    PaPr = Pa / Pr
    PaPr_dB = 10.0 * math.log10(PaPr + EPS)

# -------------------------
# Print summary
# -------------------------
print("\n--- FINAL METRICS (ROI FOR OBJECT MODE) ---")
print(f"MSE(A,T)      : {mse_AT:.4e}")
print(f"MSE(A,C)      : {mse_AC:.4e}")
print(f"NCC(A,T)      : {ncc_AT:.4f}")
print(f"NCC(A,C)      : {ncc_AC:.4f}")
print(f"PSNR(A,C)     : {psnr_AC:.2f} dB")
print(f"SSIM(A,C)     : {ssim_AC:.4f}")
print(f"PSNR(A,T)     : {psnr_AT:.2f} dB")
print(f"SSIM(A,T)     : {ssim_AT:.4f}")
print(f"Pa/Pr         : {PaPr:.4e} ({PaPr_dB:.2f} dB)")

# -------------------------
# Visualization with ROI boxes
# -------------------------
clean2d  = cln2d
target2d = tgt2d
adv2d_v  = adv2d
diff2d   = adv2d_v - clean2d

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

ax = axes[0, 0]
im = ax.imshow(clean2d, cmap="gray", origin="lower")
ax.set_title("Clean")
ax.axis("off")
draw_roi_box(ax, roi_box_clean, color="r", lw=2)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[0, 1]
im = ax.imshow(target2d, cmap="gray", origin="lower")
ax.set_title(f"Target ({target_mode})")
ax.axis("off")
draw_roi_box(ax, roi_box_target, color="g", lw=2)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 0]
im = ax.imshow(adv2d_v, cmap="gray", origin="lower")
ax.set_title("Adversarial (Attacked)")
ax.axis("off")
draw_roi_box(ax, roi_box_clean, color="r", lw=2)
draw_roi_box(ax, roi_box_target, color="g", lw=2)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 1]
im = ax.imshow(diff2d, cmap="gray", origin="lower")
ax.set_title("Diff (A - C)")
ax.axis("off")
draw_roi_box(ax, roi_box_clean, color="r", lw=2)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

plt.imshow(diff2d, cmap="gray", origin="lower")
plt.title("Diff (A - C)")
plt.axis("off")
plt.colorbar(fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()